# Solution P1 · Credit risk scoring: from raw data to the decision

**Type**: Implement with scikit-learn

## Objective

Build on your own a model to decide whether to grant a loan, following the steps of lesson 10.1: closed test set, baseline, pipeline with dirty data (missing values, high-cardinality categories, an ordinal), comparison between a linear model and XGBoost, ablation of the missingness indicators, tuning, motivated choice, one-time check on the test set, threshold
decided with the **costs** (a missed defaulter costs 8 times a wrong refusal) and explanation.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from scipy.stats import randint
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = ds.loans(n=3000, seed=4)
y = df.pop("defaulted")
X = df
NUMERIC = ["age", "income", "job_tenure", "amount"]
NOMINAL = ["city", "contract_type", "occupation"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
space = {
    "xgbclassifier__max_depth": randint(2, 6), "xgbclassifier__learning_rate": [0.02, 0.05, 0.1], "xgbclassifier__n_estimators": [100, 200, 400],
    "xgbclassifier__min_child_weight": [1, 5, 10], "xgbclassifier__subsample": [0.7, 1.0], "xgbclassifier__colsample_bytree": [0.6, 1.0], "xgbclassifier__reg_lambda": [1, 5, 20],
}


def cost(y_true, p, threshold):
    """A wrong refusal (false positive) costs 1, an approved defaulter (false negative) costs 8."""
    y_true = np.asarray(y_true)
    return int(((p >= threshold) & (y_true == 0)).sum() * 1 + ((p < threshold) & (y_true == 1)).sum() * 8)

## Commented solution

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(f"training {len(X_tr)}, test {len(X_te)} | prevalence {y_tr.mean():.3f} / {y_te.mean():.3f}")

In [ ]:
def _ordinal():
    return OrdinalEncoder(categories=[ds.EDUCATION_ORDER], handle_unknown="use_encoded_value", unknown_value=-1)


def _nominal():
    return OneHotEncoder(handle_unknown="ignore", min_frequency=15)


def linear_prep(add_indicator=True):
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=add_indicator), StandardScaler()), NUMERIC),
        ("edu", make_pipeline(_ordinal(), SimpleImputer(strategy="median", add_indicator=add_indicator), StandardScaler()), ["education"]),
        ("nom", _nominal(), NOMINAL),
    ])


def tree_prep():
    return ColumnTransformer([("num", "passthrough", NUMERIC), ("edu", _ordinal(), ["education"]), ("nom", _nominal(), NOMINAL)])

In [ ]:
def new_xgb():
    return xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, eval_metric="logloss", n_jobs=-1, random_state=0)


models = {
    "baseline": make_pipeline(linear_prep(), DummyClassifier(strategy="prior")),
    "logistic": make_pipeline(linear_prep(), LogisticRegression(max_iter=2000)),
    "xgb": make_pipeline(tree_prep(), new_xgb()),
}
cv_table = pd.DataFrame(index=list(models), columns=["auc", "auc_sd", "logloss"], dtype=float)
results = {}
for name, model in models.items():
    results[name] = cross_validate(model, X_tr, y_tr, cv=cv, scoring=["roc_auc", "neg_log_loss"])
    r = results[name]
    cv_table.loc[name] = [r["test_roc_auc"].mean(), r["test_roc_auc"].std(), -r["test_neg_log_loss"].mean()]
cv_table.round(3)

**Ablation of the missingness indicators.** `income` is missing 30% of the time among the defaulters and 6% among the others: **the fact that it is missing is a signal**. An imputer that replaces the value without flagging it throws it away.

In [ ]:
r_without = cross_validate(make_pipeline(linear_prep(add_indicator=False), LogisticRegression(max_iter=2000)), X_tr, y_tr, cv=cv, scoring="roc_auc")
auc_without_indicators = float(r_without["test_score"].mean())
indicator_loss = cv_table.loc["logistic", "auc"] - auc_without_indicators
print(f"AUC of the logistic: with indicators {cv_table.loc['logistic', 'auc']:.3f} | without {auc_without_indicators:.3f} | loss {indicator_loss:.3f}")

In [ ]:
search = RandomizedSearchCV(make_pipeline(tree_prep(), new_xgb()), space, n_iter=12, scoring="neg_log_loss", cv=cv, random_state=0, n_jobs=1)
search.fit(X_tr, y_tr)
tuned_logloss = float(-search.best_score_)
print(f"CV log-loss: base xgb {cv_table.loc['xgb', 'logloss']:.4f} | tuned xgb {tuned_logloss:.4f} | logistic {cv_table.loc['logistic', 'logloss']:.4f}")
print({k.replace('xgbclassifier__', ''): v for k, v in search.best_params_.items()})

In [ ]:
logistic_logloss_sd = float((-results["logistic"]["test_neg_log_loss"]).std())
improvement = cv_table.loc["logistic", "logloss"] - tuned_logloss
chosen = "xgb" if improvement > logistic_logloss_sd else "logistic"
chosen_model = clone(search.best_estimator_) if chosen == "xgb" else clone(models["logistic"])
print(f"improvement of tuned xgb over the logistic: {improvement:.4f} | standard deviation across folds: {logistic_logloss_sd:.4f} -> I choose: {chosen}")

**Step 7: the test set is opened once.** The choice is made; here you only measure.

In [ ]:
chosen_model.fit(X_tr, y_tr)
p_te = chosen_model.predict_proba(X_te)[:, 1]
test_auc, test_logloss = float(roc_auc_score(y_te, p_te)), float(log_loss(y_te, p_te))
print(f"test: AUC {test_auc:.3f} (in CV {cv_table.loc['logistic', 'auc']:.3f}) | log-loss {test_logloss:.3f}")

In [ ]:
p_cv = cross_val_predict(clone(chosen_model), X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
thresholds = np.linspace(0.02, 0.9, 89)
threshold = float(thresholds[int(np.argmin([cost(y_tr, p_cv, s) for s in thresholds]))])
threshold_cost, cost_05 = cost(y_te, p_te, threshold), cost(y_te, p_te, 0.5)
refuse_all_cost = int((y_te == 0).sum())
print(f"threshold chosen on the training set: {threshold:.2f} | cost on the test set: threshold {threshold:.2f} -> {threshold_cost} | threshold 0.5 -> {cost_05} | refuse everyone -> {refuse_all_cost}")

In [ ]:
pi = permutation_importance(chosen_model, X_te, y_te, scoring="roc_auc", n_repeats=10, random_state=0)
importances = pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False)
top3 = importances.index[:3].tolist()
print(importances.round(3).to_string())

In [ ]:
report = {
    "model": chosen, "cv_auc": float(cv_table.loc["logistic", "auc"]), "test_auc": test_auc, "threshold": threshold,
    "threshold_cost": threshold_cost, "refuse_all_cost": refuse_all_cost, "top3": top3,
    "notes": "1) The model is trained on a single period and a single population: it has to be recalibrated if they change. 2) The threshold depends on the assumed 8:1 cost ratio: with other costs it has to be redone.",
}
report

## Check

In [ ]:
with verify():
    check(len(X_tr) == 2250 and len(X_te) == 750 and abs(y_tr.mean() - y_te.mean()) < 0.005, "stratified 75/25 split (2250 and 750 rows, same prevalence)", "train_test_split(..., stratify=y)")
    Z = linear_prep().fit_transform(X_tr)
    Z = Z.toarray() if hasattr(Z, "toarray") else np.asarray(Z)
    A = tree_prep().fit_transform(X_tr)
    A = A.toarray() if hasattr(A, "toarray") else np.asarray(A)
    check(Z.shape[0] == 2250 and not np.isnan(Z.astype(float)).any() and A.shape[0] == 2250 and np.isnan(A.astype(float)).any() and Z.shape[1] > A.shape[1] - 5, "linear_prep leaves no NaN; tree_prep leaves them (for XGBoost)", "SimpleImputer only in the linear version")
    check(cv_table.notna().all().all() and abs(cv_table.loc["baseline", "auc"] - 0.5) < 0.02 and cv_table.loc["logistic", "auc"] > 0.8 and cv_table.loc["xgb", "auc"] > 0.79 and abs(cv_table.loc["logistic", "auc"] - cv_table.loc["xgb", "auc"]) < 0.03, "baseline at 0.5, logistic above 0.80, xgb above 0.79 and the two models within 0.03 of each other", "cross_validate with scoring=['roc_auc', 'neg_log_loss']")
    check(cv_table.loc["baseline", "logloss"] > cv_table.loc["logistic", "logloss"] + 0.1 and (cv_table.loc[["logistic", "xgb"], "auc_sd"] > 0).all(), "the baseline log-loss exceeds that of the logistic by over 0.1; positive standard deviations for the two models", "-test_neg_log_loss")
    check(auc_without_indicators is not None and indicator_loss > 0.02, "without the missingness indicators the CV AUC drops by over 0.02: the missingness carries information", "SimpleImputer(add_indicator=False)")
    check(search is not None and len(search.cv_results_["params"]) == 12 and tuned_logloss < 0.375, "12 configurations tried; log-loss of the best below 0.375", "RandomizedSearchCV(..., n_iter=12)")
    check(logistic_logloss_sd is not None and 0.005 < logistic_logloss_sd < 0.05 and chosen == "logistic", "the difference between tuned xgb and logistic is smaller than the standard deviation across folds: the logistic is chosen", "logloss(logistic) - tuned_logloss > sd?")
    check(test_auc is not None and 0.77 < test_auc < 0.86 and abs(test_auc - cv_table.loc["logistic", "auc"]) < 0.05, "test AUC between 0.77 and 0.86 and consistent with the CV one (within 0.05)", "chosen_model.fit(X_tr, y_tr) and then evaluate on the test set")
    check(threshold is not None and 0.05 <= threshold <= 0.4 and threshold_cost < 0.75 * cost_05 and threshold_cost < refuse_all_cost, "threshold between 0.05 and 0.4; cost on the test set at least 25% below that of threshold 0.5 and below refusing everyone", "argmin of cost(y_tr, p_cv, s)")
    check(len(top3) == 3 and top3[0] == "income" and "amount" in top3, "`income` is the most important feature and `amount` is among the top three", "permutation_importance on X_te, y_te")
    check(set(report) >= {"model", "cv_auc", "test_auc", "threshold", "threshold_cost", "refuse_all_cost", "top3", "notes"} and isinstance(report["notes"], str) and len(report["notes"]) > 40, "complete report, with two limits in the notes", "a dictionary with the required keys")
done()

## What you learned

- **The correct flow is a sequence of decisions, not of computations:** test set closed after the split, all the preprocessing inside the pipeline, comparison with a baseline on the same folds, choice motivated by the numbers, check only once. The test AUC (0.81) is consistent with the CV one (0.83): it is the sign that no information leaked.
- **With data of linear structure, the simple model ties XGBoost:** logistic and XGBoost are within 0.01 of AUC (0.825 and 0.818), and the tuning takes the log-loss from 0.364 to 0.363: an improvement of 0.002, much smaller than the standard deviation across folds (about 0.02). The standard error rule picks the logistic: interpretable, fast, no tuning.
- **Missingness is information:** removing the indicators from the preprocessing takes the AUC of the logistic from 0.825 to 0.788 (−0.037), because `income` is missing mostly among the defaulters. It is the kind of effect that a "clean" imputation hides. XGBoost captures it on its own with native NaN.
- **The threshold is worth more than the model:** with costs 8:1 the optimal threshold is about 0.09, far below the 0.5 of habit: the cost on the test set drops from 800 to 446 (−44%) and also beats refusing everyone (603). With threshold 0.5 the model costs **more** than granting no loans at all.
- **The honest explanation is done on the test set:** `income` alone is worth 0.13 of AUC, followed by `amount` and `occupation` (0.04 each). The report also has to include the limits: dependence on the assumed costs and on a single training population.

## Common pitfall

Choosing the model by looking at the test set, or choosing the most complex one because "it has the lowest log-loss" even when the difference is much smaller than the variability across folds. If the difference is in the noise, simplicity wins.

## Interview question

<details><summary>How do you choose between a linear model and XGBoost when the CV results are almost equal?</summary>

I look at the difference compared with the standard deviation across the folds (or I use a repeated CV): if it is within the noise, there is no evidence that the complex one is better, so I choose the simple one for interpretability, training speed and stability. I choose the complex one only if the advantage exceeds the standard error or if there is a domain reason
(known interactions, a lot of data). The decision is then evaluated with the costs, not with the metric alone.

</details>